# SDMX 3.2: from the SDMX definition of a VTL program to its result

**Eurostat's GDP check**, the example of the presentation *Simplify the SDMX -
VTL interconnection* (SDMX Experts Workshop 2026), on real data.

| Part | Question | SDMX 3.2 artefact |
|---|---|---|
| Structural part | What does it compute? | `TransformationScheme` |
| Structural part | On which structures? | `VtlMappingScheme` |
| Execution model | How is it run? | `VtlExecution` |

`pysdmx` handles the SDMX artefacts and the queries to the web services, and
`vtlengine` runs the VTL. SDMX 3.2 is a draft: its support in `pysdmx` is in
the pull requests [#760](https://github.com/bis-med-it/pysdmx/pull/760),
[#768](https://github.com/bis-med-it/pysdmx/pull/768),
[#762](https://github.com/bis-med-it/pysdmx/pull/762) and
[#770](https://github.com/bis-med-it/pysdmx/pull/770) (branch
`sdmx_experts_prep`, version 1.24.0rc1, with the `all` extras).

The structures and the data come from the web services of Eurostat and the
OECD, so the notebook needs access to the Internet. The Dataflow of the
results, the identifiers of the VTL artefacts and the web service of the
national institute are illustrative.

This course covers:

- how a VTL program and its mappings are defined as SDMX artefacts;
- how an execution says where the input data come from;
- how the definitions are checked and exchanged in SDMX-ML and SDMX-JSON;
- how an engine goes from the reference to an execution to the stored
  result;
- how a Codelist becomes a VTL Value Domain, and how a query extraction
  and a URL extraction compare.

| # | Step |
|---|------|
| 1 | Setup |
| 2 | The input: a real Dataflow |
| 3 | Structural part: the program and its mappings |
| 4 | Execution model: where the input data come from |
| 5 | Checking and exchanging the definitions |
| 6 | At run time |
| 7 | A VTL Value Domain from a Codelist |
| 8 | Query or URL extraction |
| 9 | Recap |

> **Requirements:** the pysdmx wheel and `helpers.py` in the folder of
> this notebook, and access to the Internet.

> **Practice:** `sdmx_vtl_practice.ipynb` is the exercise on Sections 2
> to 6.

## 1 · Setup

`pysdmx` and `vtlengine` are installed and the required names are
imported:

- the classes of the VTL artefacts (`TransformationScheme`,
  `VtlMappingScheme`, `VtlExecution` and their items) are in
  `pysdmx.model`;
- `pysdmx.toolkit.vtl` converts between SDMX and VTL, and checks the
  definitions;
- `read_sdmx` and `write_sdmx` read and write the SDMX messages;
- `run` and `semantic_analysis` are the two entry points of `vtlengine`;
- `helpers.py`, in the folder of this notebook, has the address of the
  web service of Eurostat (`EUROSTAT`) and the functions that turn a
  data extraction into a query (`data_query`) and into its data
  (`retrieve`).

In [1]:
%pip install --quiet ./pysdmx-1.24.0rc1-py3-none-any.whl[all]

import json

import pandas as pd
from helpers import EUROSTAT, data_query, retrieve
from msgspec.structs import replace
from vtlengine import run, semantic_analysis

from pysdmx.io import read_sdmx, write_sdmx
from pysdmx.io.format import Format
from pysdmx.io.pd import PandasDataset
from pysdmx.model import (
    ItemReference,
    QueryExtraction,
    QueryParameter,
    Reference,
    Transformation,
    TransformationScheme,
    UrlExtraction,
    VtlCodelistMapping,
    VtlDatasetMapping,
    VtlExecution,
    VtlMappingScheme,
)
from pysdmx.toolkit.vtl import (
    convert_codelist_to_vtl,
    convert_dataset_to_sdmx,
    convert_dataset_to_vtl,
    generate_vtl_script,
    model_validations,
)
from pysdmx.util import parse_urn

## 2 · The input: a real Dataflow

The Dataflow `ESTAT:NAMA_10_GDP(1.0)` (GDP and main components) and its Data
Structure Definition are retrieved from Eurostat.

In [2]:
message = read_sdmx(
    f"{EUROSTAT}/structure/dataflow/ESTAT/NAMA_10_GDP/1.0"
    "?references=descendants"
)
dsd_gdp = message.get_data_structure_definitions()[0]
df_gdp = replace(message.get_dataflows()[0], structure=dsd_gdp)
print(df_gdp.short_urn, [c.id for c in dsd_gdp.components])

Dataflow=ESTAT:NAMA_10_GDP(1.0) ['freq', 'unit', 'na_item', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']


## 3 · Structural part: the program and its mappings

The Transformation Scheme is the VTL program: it computes the annual growth of
GDP and keeps the data points whose growth is above 20% in absolute value.

Its VTL Mapping Scheme defines the VTL Data Sets of the program: each
`VtlDatasetMapping` has the alias used in VTL, the SDMX artefact and
`is_input`. The output is the Dataflow `ESTAT:ES_NAMA_10_GDP(1.0)`, which is
not defined beforehand: its data structure is derived from the program.

In [3]:
mappings = VtlMappingScheme(
    id="VTL_MAP_GDP",
    agency="ESTAT",
    name="VTL mappings of the GDP check",
    items=[
        VtlDatasetMapping(
            id="GDP",
            name="GDP",
            vtl_alias="gdp",
            sdmx_artefact=df_gdp,
            is_input=True,
        ),
        VtlDatasetMapping(
            id="GDP_CHECK",
            name="Validation results",
            vtl_alias="gdp_check",
            sdmx_artefact=Reference(
                "Dataflow", "ESTAT", "ES_NAMA_10_GDP", "1.0"
            ),
            is_input=False,
        ),
    ],
)
ts = TransformationScheme(
    id="TS_GDP_CHECK",
    agency="ESTAT",
    name="GDP check",
    vtl_version="2.2",
    vtl_mapping_scheme=mappings,
    items=[
        Transformation(
            id="GROWTH",
            name="Annual growth of GDP (%)",
            result="growth",
            is_persistent=False,
            expression="(gdp / timeshift(gdp, 1) - 1) * 100",
        ),
        Transformation(
            id="GDP_CHECK",
            name="Growth of GDP within 20%",
            result="gdp_check",
            is_persistent=True,
            expression=(
                'check(abs(growth) <= 20 errorcode "GDP_GROWTH" invalid)'
            ),
        ),
    ],
)

pd.DataFrame(
    [(m.vtl_alias, m.sdmx_urn, m.is_input) for m in mappings.items],
    columns=["vtlAlias", "sdmxUrn", "isInput"],
)

,vtlAlias,sdmxUrn,isInput
0,gdp,urn:sdmx:org.sdmx.infomodel.datastructure.Data...,True
1,gdp_check,urn:sdmx:org.sdmx.infomodel.datastructure.Data...,False


> **Key point:** the Transformation Scheme and its VTL Mapping Scheme say what is computed and on which structures. They do not say where the data come from.

## 4 · Execution model: where the input data come from

A `VtlExecution` references the Transformation Scheme and has at most one data
extraction for each input. It is maintained independently of the program:
Eurostat queries its own web service for all countries, and a national
institute reuses the program with its own execution, which queries its own
repository for its country. The key is fixed, and the period is a default that
can be changed at run time.

In [4]:
def gdp_check_execution(agency, entry_point, key):
    """Returns an execution of the GDP check."""
    return VtlExecution(
        id="EXEC_GDP_CHECK",
        agency=agency,
        name=f"Execution of the GDP check by {agency}",
        transformation_scheme=ts,
        extractions=[
            QueryExtraction(
                dataset_mapping=ItemReference(
                    "VtlDatasetMapping", "ESTAT", "VTL_MAP_GDP", "1.0", "GDP"
                ),
                entry_point=entry_point,
                sdmx_api_version="2.0",
                parameters=[
                    QueryParameter(name="key", value=key, is_fixed=True),
                    QueryParameter(
                        name="c[TIME_PERIOD]", value="ge:2015", is_fixed=False
                    ),
                ],
            )
        ],
    )


exec_estat = gdp_check_execution("ESTAT", EUROSTAT, "A.CP_MEUR.B1GQ.*")
exec_ine = gdp_check_execution(
    "INE", "https://sdmx.nsi.example/rest", "A.CP_MEUR.B1GQ.ES"
)

> **Key point:** an execution is a separate artefact, so the same program is run by different organisations, each with its own data extraction.

## 5 · Checking and exchanging the definitions

`model_validations` checks the rules that involve several artefacts (e.g. the
outputs are persistent results, the extractions apply to inputs). The VTL
artefacts are written and read in SDMX-ML 3.2 and SDMX-JSON 2.2, validated
against the draft schemas.

In [5]:
vtl_artefacts = [mappings, ts, exec_estat, exec_ine]
for artefact in vtl_artefacts:
    model_validations(artefact, [df_gdp, *vtl_artefacts])

sdmx_ml = write_sdmx(vtl_artefacts, Format.STRUCTURE_SDMX_ML_3_2)
sdmx_json = write_sdmx(vtl_artefacts, Format.STRUCTURE_SDMX_JSON_2_2_0)
print(
    len(read_sdmx(sdmx_ml).structures),
    "artefacts read from SDMX-ML 3.2 and",
    len(read_sdmx(sdmx_json).structures),
    "from SDMX-JSON 2.2",
)
start = sdmx_ml.index("<str:VtlExecution ")
print(sdmx_ml[start : sdmx_ml.index("</str:VtlExecution>", start)])

4 artefacts read from SDMX-ML 3.2 and 4 from SDMX-JSON 2.2
<str:VtlExecution id="EXEC_GDP_CHECK" version="1.0" isExternalReference="false" agencyID="ESTAT">
				<com:Name xml:lang="en">Execution of the GDP check by ESTAT</com:Name>
				<str:TransformationScheme>urn:sdmx:org.sdmx.infomodel.transformation.TransformationScheme=ESTAT:TS_GDP_CHECK(1.0)</str:TransformationScheme>
				<str:QueryExtraction sdmxApiVersion="2.0">
					<str:DatasetMapping>urn:sdmx:org.sdmx.infomodel.transformation.VtlDatasetMapping=ESTAT:VTL_MAP_GDP(1.0).GDP</str:DatasetMapping>
					<str:EntryPoint>https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0</str:EntryPoint>
					<str:QueryParameter name="key" value="A.CP_MEUR.B1GQ.*" isFixed="true"/>
					<str:QueryParameter name="c[TIME_PERIOD]" value="ge:2015" isFixed="false"/>
				</str:QueryExtraction>
			


> **Key point:** `model_validations` raises an error when a rule that involves several artefacts is not met, before anything is run or exchanged.

## 6 · At run time

The engine needs only the reference to the `VtlExecution` and, possibly, some
parameter values.

### Steps 1 and 2: the execution, the program and its data structures

The VTL script is generated from the Transformation Scheme. The semantic
analysis of the engine gets the SDMX artefact and the alias of the input, and
gives the data structure of every result without any data. The one of the
output is converted into SDMX: these are the components of the Dataflow
`ESTAT:ES_NAMA_10_GDP(1.0)`.

In [6]:
execution, values = exec_estat, {"c[TIME_PERIOD]": "ge:2015"}
program = execution.transformation_scheme
gdp_mapping, check_mapping = program.vtl_mapping_scheme.items

script = generate_vtl_script(program, model_validation=True)
print(script)

analysis = semantic_analysis(
    script=program,
    data_structures=df_gdp,
    sdmx_mappings={df_gdp.short_urn: gdp_mapping.vtl_alias},
)
schema_check = convert_dataset_to_sdmx(
    analysis[check_mapping.vtl_alias],
    reference=parse_urn(check_mapping.sdmx_urn),
).structure

pd.DataFrame(
    [(c.id, c.role.name, str(c.dtype)) for c in schema_check.components],
    columns=["component", "role", "data type"],
)

growth := (gdp / timeshift(gdp, 1) - 1) * 100;
gdp_check <- check(abs(growth) <= 20 errorcode "GDP_GROWTH" invalid);



,component,role,data type
0,freq,DIMENSION,String
1,unit,DIMENSION,String
2,na_item,DIMENSION,String
3,geo,DIMENSION,String
4,TIME_PERIOD,DIMENSION,ObservationalTimePeriod
5,bool_var,MEASURE,Boolean
6,imbalance,MEASURE,Double
7,errorcode,MEASURE,String
8,errorlevel,MEASURE,Integer


### Step 3: queries the data of the input

The query is built from the extraction: the artefact of the mapping (or the
context), the `key` and the other parameters, which take the run-time values
unless they are fixed. A `QueryExtraction` becomes a `DataQuery` of `pysdmx`,
sent with a `LenientRestService`; in a `UrlExtraction`, the tokens of the
template are replaced.

`data_query` and `retrieve` are in `helpers.py`: the first one returns the
SDMX REST version and the `DataQuery` of an extraction, and the second one
sends it, prints its address and returns the data.

In [7]:
gdp = PandasDataset(
    structure=replace(
        dsd_gdp.to_schema(),
        context="dataflow",
        id=df_gdp.id,
        version=df_gdp.version,
    ),
    data=retrieve(execution.extractions[0], gdp_mapping.sdmx_urn, values),
)
print(len(gdp.data), "observations of", gdp.data["geo"].nunique(), "areas")

# The national institute would query its own repository (illustrative)
version, query = data_query(exec_ine.extractions[0], gdp_mapping.sdmx_urn)
print(
    exec_ine.extractions[0].entry_point
    + query.get_url(version, omit_defaults=True)
)

https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/NAMA_10_GDP/1.0/A.CP_MEUR.B1GQ.*?c[TIME_PERIOD]=ge:2015
504 observations of 46 areas
https://sdmx.nsi.example/rest/data/dataflow/ESTAT/NAMA_10_GDP/1.0/A.CP_MEUR.B1GQ.ES?c[TIME_PERIOD]=ge:2015


### Step 4: executes the program

The SDMX data set is converted into a VTL Data Set, named after its alias,
and the engine runs the script on it.

In [8]:
gdp_vtl = convert_dataset_to_vtl(gdp, gdp_mapping.vtl_alias)
results = run(
    script=script,
    data_structures=json.loads(gdp_vtl.to_json_datastructure()),
    datapoints={gdp_vtl.name: gdp_vtl.data},
    return_only_persistent=False,
)

# The five largest growth rates
growth = results["growth"].data
growth.loc[growth["OBS_VALUE"].abs().nlargest(5).index].round(1)

,freq,unit,na_item,geo,TIME_PERIOD,OBS_VALUE
355,A,CP_MEUR,B1GQ,NO,2022,33.3
354,A,CP_MEUR,B1GQ,NO,2021,31.3
246,A,CP_MEUR,B1GQ,IS,2022,25.1
435,A,CP_MEUR,B1GQ,TR,2022,24.5
442,A,CP_MEUR,B1GQ,UA,2019,24.1


### Step 5: stores the result

The persistent result `gdp_check` is the output of the program: its data are
converted into SDMX, as data of the Dataflow `ESTAT:ES_NAMA_10_GDP(1.0)`. The
conversion checks that its data structure is the one obtained in Step 2.

In [9]:
checked = convert_dataset_to_sdmx(
    results[check_mapping.vtl_alias], schema=schema_check
)
print(write_sdmx(checked, Format.DATA_SDMX_CSV_2_0_0))

STRUCTURE,STRUCTURE_ID,ACTION,freq,unit,na_item,geo,TIME_PERIOD,bool_var,imbalance,errorcode,errorlevel
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,AL,2023,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,BG,2022,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,IS,2022,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,ME,2022,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,NO,2021,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,NO,2022,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,NO,2023,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,TR,2022,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,UA,2019,False,,GDP_GROWTH,
dataflow,ESTAT:ES_NAMA_10_GDP(1.0),I,A,CP_MEUR,B1GQ,UA,2021,False,,GDP_GROWTH,



> **Key point:** the engine starts from the reference to a `VtlExecution`. The data structure of the output is known before any data are read, and the stored data are checked against it.

## 7 · A VTL Value Domain from a Codelist

A `VtlCodelistMapping` maps a VTL Value Domain to a Codelist, whose Codes are
its values. Here the alias `cl_eu` stands for the Codelist of `geo` restricted
to the areas of the European Union: Eurostat returns this partial Codelist
with a Dataflow that only has those areas, the one of the government deficit
and debt (`detail=referencepartial`). A VTL statement keeps their data points.

In [10]:
eu_message = read_sdmx(
    f"{EUROSTAT}/structure/dataflow/ESTAT/GOV_10DD_EDPT1/1.0"
    "?detail=referencepartial&references=descendants"
)
cl_eu = next(cl for cl in eu_message.get_codelists() if cl.id == "GEO")
print(eu_message.get_dataflows()[0].name, "->", len(cl_eu.codes), "areas")

eu_mapping = VtlCodelistMapping(
    id="CL_EU",
    name="Areas of the European Union",
    vtl_alias="cl_eu",
    codelist=cl_eu,
)
# It belongs to the VTL Mapping Scheme of the program that uses it
model_validations(replace(mappings, items=[*mappings.items, eu_mapping]))
print(eu_mapping.vtl_alias, "->", eu_mapping.sdmx_urn)

# The Codelist is converted into a VTL Value Domain, named after its alias
value_domain = convert_codelist_to_vtl(cl_eu, eu_mapping.vtl_alias)
# The engine takes its Value Domains as dictionaries
value_domains = value_domain.to_dict()

gdp_eu = run(
    script="gdp_eu <- gdp [filter geo in cl_eu];",
    data_structures=json.loads(gdp_vtl.to_json_datastructure()),
    datapoints={gdp_vtl.name: gdp_vtl.data},
    value_domains=value_domains,
)["gdp_eu"].data
print(
    len(gdp_eu),
    "of",
    len(gdp_vtl.data),
    "data points are of",
    gdp_eu["geo"].nunique(),
    "areas of the European Union; the others are of",
    ", ".join(sorted(set(gdp_vtl.data["geo"]) - set(gdp_eu["geo"]))),
)

Government deficit/surplus, debt and associated data -> 31 areas
cl_eu -> urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:GEO(28.0)
341 of 504 data points are of 31 areas of the European Union; the others are of AL, BA, CH, EA, EA12, IS, LI, MD, ME, MK, NO, RS, TR, UA, XK


## 8 · Query or URL extraction

Two ways to retrieve a past edition of Spain's quarterly GDP from the OECD
revisions database, which keeps every monthly edition as a Dimension. At run
time, the edition 202605 (May 2026) is given instead of the default one, and
both forms return the same data.

In [11]:
REVISIONS = (
    "urn:sdmx:org.sdmx.infomodel.datastructure.Dataflow="
    "OECD.SDD.STES:DSD_STES_REVISIONS@DF_STES_REVISIONS(4.0)"
)
mapping = ItemReference(
    "VtlDatasetMapping", "OECD.SDD.STES", "VTL_MAP", "1.0", "GDP"
)
by_query = QueryExtraction(
    dataset_mapping=mapping,
    entry_point="https://sdmx.oecd.org/public/rest/v2",
    sdmx_api_version="2.2",
    parameters=[
        QueryParameter(
            name="key", value="ESP.Q.B1GQ_Q.XDC._T.*", is_fixed=True
        ),
        QueryParameter(name="c[EDITION]", value="202610", is_fixed=False),
    ],
)
by_url = UrlExtraction(
    dataset_mapping=mapping,
    url_template=(
        "https://sdmx.oecd.org/public/rest/data/"
        "OECD.SDD.STES,DSD_STES_REVISIONS@DF_STES_REVISIONS,4.0/"
        "ESP.Q.B1GQ_Q.XDC._T.{edition}"
    ),
    parameters=[
        QueryParameter(name="edition", value="202610", is_fixed=False)
    ],
)

columns = ["TIME_PERIOD", "OBS_VALUE"]
from_query = retrieve(by_query, REVISIONS, {"c[EDITION]": "202605"})
from_url = retrieve(by_url, REVISIONS, {"edition": "202605"})
from_query = from_query[columns].sort_values("TIME_PERIOD", ignore_index=True)
from_url = from_url[columns].sort_values("TIME_PERIOD", ignore_index=True)
print("Same data:", from_query.astype(str).equals(from_url.astype(str)))
from_query.tail(3)

https://sdmx.oecd.org/public/rest/v2/data/dataflow/OECD.SDD.STES/DSD_STES_REVISIONS@DF_STES_REVISIONS/4.0/ESP.Q.B1GQ_Q.XDC._T.*?c[EDITION]=202605
https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES_REVISIONS@DF_STES_REVISIONS,4.0/ESP.Q.B1GQ_Q.XDC._T.202605
Same data: True


,TIME_PERIOD,OBS_VALUE
122,2025-Q3,350169000000
123,2025-Q4,352828000000
124,2026-Q1,354993000000


## 9 · Recap

| Step | `pysdmx` and `vtlengine` |
|---|---|
| Define the program and its mappings | `TransformationScheme`, `VtlMappingScheme`, `VtlDatasetMapping` |
| Define how it is run | `VtlExecution`, `QueryExtraction`, `UrlExtraction`, `QueryParameter` |
| Check the definitions | `model_validations` |
| Exchange the definitions | `write_sdmx` and `read_sdmx` (SDMX-ML 3.2, SDMX-JSON 2.2) |
| Generate the VTL script | `generate_vtl_script` |
| Derive the data structure of the output | `semantic_analysis` (`vtlengine`), `convert_dataset_to_sdmx` |
| Query the data of the input | `DataQuery`, `LenientRestService`, `read_sdmx` |
| Execute the program | `convert_dataset_to_vtl`, `run` (`vtlengine`) |
| Store the result | `convert_dataset_to_sdmx`, `write_sdmx` |
| Define a VTL Value Domain | `VtlCodelistMapping`, `convert_codelist_to_vtl` |

- The program (`TransformationScheme`) and its mappings
  (`VtlMappingScheme`) are the structural part: they do not say where the
  data come from.
- A `VtlExecution` is maintained independently of the program, so the
  same program is reused with different executions.
- The data structure of the output is derived from the program, without
  any data.
- At run time, the engine needs only the reference to the `VtlExecution`
  and, possibly, some parameter values.

### Practice

`sdmx_vtl_practice.ipynb` is the exercise on Sections 2 to 6: five of its
steps are to be completed by the participant. The same exercise is
available as a script, `sdmx_vtl_practice.py`. The answer is in the
`solution` folder, as a notebook and as a script.